# 02 - Training Pipeline Design and Mondrian Conformal Calibration

This notebook is the second stage of the pipeline.

It consumes the preprocessed output from `01_eda_and_problem_setup.ipynb` and runs:
- Direct multi-horizon model training (96-step horizon)
- Mondrian conformal calibration with hierarchical fallbacks
- Holdout evaluation (point + interval metrics)
- A small inference demo with latest forecasts

All paths are local and artifact outputs are saved to timestamped run directories.

For the public repository, this notebook is mainly a readable training/design artifact rather than the recommended quickstart path.


### Demo Service Link (Runtime Context)

The FastAPI + Dash demo service in `project2/app` and `project2/dashboard` consumes the sanitized artifact at `project2/mondrian_artifacts_demo` and local data from `project2/df_stationsv3.csv`.

Active runtime station scope is fixed to: `Station_1`, `Station_2`, `Station_3`, `Station_4`, `Station_7`, `Station_8`. Runtime manual calibration overlays are written only to `project2/runtime/calibration_states/` so training artifacts remain immutable.


## 1) Setup and Path Resolution

Run `01_eda_and_problem_setup.ipynb` first so `project2/data/processed/df_pre.parquet` exists.


In [ ]:
from __future__ import annotations

import json
import time as tm
from collections import defaultdict
from io import BytesIO
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder, RobustScaler

try:
    import xgboost as xgb
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "xgboost is required for this notebook. Install it with `pip install xgboost`."
    ) from exc


def render_figure(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    display(Image(data=buf.getvalue()))
    plt.close(fig)


def is_project2_root(path: Path) -> bool:
    return (path / "app" / "main.py").exists() and (path / "scripts" / "run_demo_api.py").exists()


def resolve_project2_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        if base.name == "project2" and is_project2_root(base):
            return base
        nested = base / "project2"
        if is_project2_root(nested):
            return nested
    raise FileNotFoundError(
        "Could not locate project2 root. Run this notebook from GestaguaOG or project2."
    )


PROJECT2_ROOT = resolve_project2_root()
PROCESSED_PATH = PROJECT2_ROOT / "data" / "processed" / "df_pre.parquet"
RUNS_ROOT = PROJECT2_ROOT / "mondrian_artifacts" / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT2_ROOT}")
print(f"Preprocessed input: {PROCESSED_PATH}")
print(f"Run artifacts root: {RUNS_ROOT}")


## 2) Locked Configuration

The public training flow is hard-locked to horizon 96 and your recommended parameters.


In [ ]:
CFG = {
    "in_length": 96,
    "horizon": 96,
    "step_size": 4,
    "alpha": 0.10,
    "beta": 0.90,
    "min_bin_n": 100,
    "n_blocks": 2,
    "min_train_days": 7,
    "tz": "Europe/Madrid",
    "holdout_frac": 0.05,
    "xgb_params": {
        "tree_method": "hist",
        "random_state": 42,
        "max_depth": 9,
        "learning_rate": 0.03823968436952936,
        "n_estimators": 370,
        "subsample": 0.801704069232722,
        "colsample_bytree": 0.789775025458991,
        "gamma": 0.000898038425726333,
        "reg_alpha": 2.7107447695850864,
        "reg_lambda": 2.0591320678176355,
    },
}

BASE_FEATURES = [
    "mean_1h",
    "med_6h",
    "std_6h",
    "lag_96",
    "diff_96",
    "lag96_dow_interact",
    "dow_sin",
    "dow_cos",
    "max_24h",
    "std_24h",
]

if CFG["horizon"] != 96:
    raise ValueError(
        f"Invalid configuration: horizon must be 96, got {CFG['horizon']}."
    )

print("Locked config loaded. Horizon is fixed to:", CFG["horizon"])
print("XGBoost params:")
print(json.dumps(CFG["xgb_params"], indent=2))


## 3) Load Preprocessed Data


In [ ]:
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(
        "Missing preprocessed input. Run 01_eda_and_problem_setup.ipynb first to create "
        "project2/data/processed/df_pre.parquet."
    )


df_pre = pd.read_parquet(PROCESSED_PATH)
required_cols = {"timestamp", "station", "consumption", "consumption_clean", "zscore_pos", "is_leak"}
missing = required_cols - set(df_pre.columns)
if missing:
    raise KeyError(f"df_pre is missing required columns: {sorted(missing)}")

# Normalize dtypes and sort
df_pre = df_pre.copy()
df_pre["timestamp"] = pd.to_datetime(df_pre["timestamp"], utc=True, errors="coerce")
df_pre = df_pre.dropna(subset=["timestamp", "consumption_clean"]).copy()
df_pre["station"] = df_pre["station"].astype(str)
df_pre["consumption_clean"] = pd.to_numeric(df_pre["consumption_clean"], errors="coerce")
df_pre = df_pre.dropna(subset=["consumption_clean"])

# Synthetic-only public scope
df_pre = df_pre[df_pre["station"].str.startswith("Station_")].copy()
df_pre = df_pre.sort_values(["station", "timestamp"], kind="mergesort").reset_index(drop=True)

print("Rows:", len(df_pre))
print("Stations:", df_pre["station"].nunique())
print("Timestamp range:", df_pre["timestamp"].min(), "->", df_pre["timestamp"].max())
df_pre.head()


## 4) Time Splits (Train / Validation / Holdout)

We use a strict chronological split **per station** to respect forecasting causality.

1. **Development vs Holdout split**
- The latest 5% of each station is reserved as holdout.
- This simulates future performance after all design choices are fixed.

2. **Train vs Validation split inside Development**
- The latest slice of development data is used as fold validation material for calibration.
- Training and validation remain temporally ordered for each station.

Why this matters:
- It prevents training on future information.
- It keeps station-level behavior intact.
- It aligns with real deployment where forecasts are made only from past data.


In [ ]:
def split_tail_per_station(
    df: pd.DataFrame,
    tail_frac: float,
    min_tail_rows: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_parts = []
    tail_parts = []

    for station, g in df.groupby("station", sort=False):
        g = g.sort_values("timestamp").reset_index(drop=True)
        n = len(g)
        if n < (min_tail_rows + CFG["in_length"] + CFG["horizon"]):
            # Not enough history for safe holdout windows; keep all in train.
            train_parts.append(g)
            continue

        tail_n = max(min_tail_rows, int(round(n * tail_frac)))
        tail_n = min(tail_n, n - (CFG["in_length"] + CFG["horizon"]))
        split_idx = n - tail_n

        train_parts.append(g.iloc[:split_idx].copy())
        tail_parts.append(g.iloc[split_idx:].copy())

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else df.iloc[0:0].copy()
    tail_df = pd.concat(tail_parts, ignore_index=True) if tail_parts else df.iloc[0:0].copy()
    return train_df, tail_df


# Outer split: development + holdout
df_dev, df_holdout = split_tail_per_station(
    df_pre,
    tail_frac=CFG["holdout_frac"],
    min_tail_rows=CFG["horizon"] * 3,
)

# Inner split: train + validation for calibration CV/final fit
df_train, df_val = split_tail_per_station(
    df_dev,
    tail_frac=0.05,
    min_tail_rows=CFG["horizon"] * 2,
)

print("Development rows:", len(df_dev))
print("Train rows:", len(df_train))
print("Validation rows:", len(df_val))
print("Holdout rows:", len(df_holdout))

if df_holdout.empty:
    raise ValueError("Holdout split is empty. Increase data size or reduce holdout fraction.")


## 5) Feature Engineering Helpers


In [ ]:
def _make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)


def fit_scalers_per_station(df_train_fold: pd.DataFrame) -> dict[str, RobustScaler | None]:
    scalers: dict[str, RobustScaler | None] = {}
    for st, g in df_train_fold.groupby("station"):
        vals = pd.to_numeric(g["consumption_clean"], errors="coerce").dropna()
        if len(vals) >= 24:
            rs = RobustScaler()
            rs.fit(vals.to_frame().values)
            scalers[st] = rs
        else:
            scalers[st] = None
    return scalers


def apply_scalers(df: pd.DataFrame, scalers: dict[str, RobustScaler | None]) -> pd.DataFrame:
    out = []
    for st, g in df.groupby("station", sort=False):
        g = g.copy()
        rs = scalers.get(st)
        if rs is None:
            g["cons_scaled"] = g["consumption_clean"].astype(float).values
        else:
            g["cons_scaled"] = rs.transform(g[["consumption_clean"]].values).ravel()
        out.append(g)
    return pd.concat(out, ignore_index=True) if out else df.iloc[0:0].copy()


def fit_station_onehot(df_train_fold: pd.DataFrame, col: str = "station", prefix: str = "st__"):
    ohe = _make_ohe()
    arr = ohe.fit_transform(df_train_fold[[col]])
    cols = [f"{prefix}{c}" for c in ohe.categories_[0].tolist()]
    train_ohe = pd.DataFrame(arr, columns=cols, index=df_train_fold.index)
    return ohe, cols, train_ohe


def apply_station_onehot(df_any: pd.DataFrame, ohe, cols: list[str], col: str = "station", prefix: str = "st__"):
    arr = ohe.transform(df_any[[col]])
    got = [f"{prefix}{c}" for c in ohe.categories_[0].tolist()]
    x_ohe = pd.DataFrame(arr, columns=got, index=df_any.index)
    for c in cols:
        if c not in x_ohe:
            x_ohe[c] = 0.0
    x_ohe = x_ohe[cols]
    return pd.concat([df_any, x_ohe], axis=1)


def add_calendar_features(g: pd.DataFrame, tz: str = "Europe/Madrid") -> pd.DataFrame:
    g = g.sort_values("timestamp").copy()
    ts_loc = pd.to_datetime(g["timestamp"], utc=True).dt.tz_convert(tz)

    g["dow_local"] = ts_loc.dt.weekday
    g["hour_local"] = ts_loc.dt.hour
    g["minute_local"] = ts_loc.dt.minute

    minutes_local = g["hour_local"] * 60 + g["minute_local"]
    g["tod_sin"] = np.sin(2 * np.pi * minutes_local / (24 * 60))
    g["tod_cos"] = np.cos(2 * np.pi * minutes_local / (24 * 60))
    g["dow_sin"] = np.sin(2 * np.pi * g["dow_local"] / 7.0)
    g["dow_cos"] = np.cos(2 * np.pi * g["dow_local"] / 7.0)
    return g


def add_target_features_in_time(g: pd.DataFrame, value_col: str = "cons_scaled") -> pd.DataFrame:
    if value_col not in g.columns:
        raise KeyError(f"'{value_col}' not found. Apply scaling first.")

    g = g.sort_values("timestamp").copy()
    t = pd.to_numeric(g[value_col], errors="coerce").astype("float32")

    g["lag_96"] = t.shift(96)
    g["diff_96"] = t - g["lag_96"]

    r_4 = t.rolling(window=4, min_periods=1)
    g["mean_1h"] = r_4.mean()

    r_24 = t.rolling(window=24, min_periods=1)
    g["med_6h"] = r_24.median()
    g["std_6h"] = r_24.std()

    r_96 = t.rolling(window=96, min_periods=1)
    g["max_24h"] = r_96.max()
    g["std_24h"] = r_96.std()

    g["lag96_dow_interact"] = g["lag_96"] * g["dow_cos"]

    # conservative missing handling for model matrix stability
    for col in ["mean_1h", "med_6h", "std_6h", "lag_96", "diff_96", "lag96_dow_interact", "dow_sin", "dow_cos", "max_24h", "std_24h"]:
        g[col] = pd.to_numeric(g[col], errors="coerce").astype("float32")

    return g


def ensure_is_bad(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    leak = pd.to_numeric(out.get("is_leak", 0), errors="coerce").fillna(0).astype(int).ne(0)
    miss = pd.to_numeric(out["consumption_clean"], errors="coerce").isna()
    out["is_bad"] = (leak | miss).astype("int8")
    return out


## 6) Windowing, Mondrian Keys, and Model Utilities

This section contains the core mechanics that convert station time series into supervised learning samples and uncertainty buckets.

### A) Windowing design (supervised sample builder)

For each station, we build sliding windows on the scaled target:

- `in_length = 96`: the model input history (96 x 15 min = 24 hours).
- `horizon = 96`: direct multi-step target (next 24 hours).
- `step_size = 4`: we keep one window every hour to control sample overlap and runtime.

Each valid window generates:
- `X`: flattened lag/rolling/calendar features from the input history.
- `Y`: 96-step future target vector.
- `T0`: forecast start timestamp.
- `S`: station identifier.
- `T`: timestamp matrix for each horizon step.

### B) Window quality control

A window is kept only if it is reliable end-to-end:

- Bad-flag masking is applied over the **full block** (`in_length + horizon`).
- Non-finite targets are rejected.
- Optional non-finite feature rows are rejected.

This creates a conservative training set that avoids contaminating calibration with problematic slices.

### C) Mondrian key construction

For every forecasted point `(window, horizon_step)`, we create a key:

- `(station, horizon_index, local_time_of_day_bin)`

Where:
- `station` captures infrastructure-specific behavior.
- `horizon_index` captures lead-time difficulty (short-term vs long-term).
- `time_of_day_bin` captures intraday heteroscedasticity.

This key design allows interval widths to adapt to both **who** is being forecast and **when** in the day/horizon we are forecasting.

### D) Fallback logic for interval lookup

If a very specific key is sparse, the calibrator backs off in this order:

1. `(station, horizon, time_of_day)`
2. `(station, horizon)`
3. `(station,)`
4. global

So the pipeline keeps granular uncertainty where data is dense, while staying robust in sparse bins.


In [ ]:
def prepare_xy_pooled_with_time(
    df: pd.DataFrame,
    features: list[str],
    target_col: str,
    in_len: int,
    horizon: int,
    step_size: int,
    bad_flag_cols: tuple[str, ...] = ("is_bad",),
    require_non_nan_features: bool = True,
):
    x_list, y_list, t0_list, s_list, t_list = [], [], [], [], []

    for st, sub in df.groupby("station", sort=False):
        sub = sub.sort_values("timestamp", kind="mergesort")

        feat = sub[features].to_numpy(dtype=np.float32, copy=False)
        targ = pd.to_numeric(sub[target_col], errors="coerce").to_numpy(dtype=np.float32, copy=False)
        ts_vals = pd.to_datetime(sub["timestamp"], utc=True).astype("int64").to_numpy(copy=False)

        n = len(sub)
        need = in_len + horizon
        if n < need:
            continue

        present = [c for c in bad_flag_cols if c in sub.columns]
        if present:
            bad_any_row = sub[present].any(axis=1).to_numpy(dtype=bool, copy=False)
        else:
            bad_any_row = np.zeros(n, dtype=bool)

        targ_bad = ~np.isfinite(targ)
        bad_any_row |= targ_bad

        feat_win = sliding_window_view(feat, window_shape=(in_len, feat.shape[1]))[:, 0, :, :]
        fut_y = sliding_window_view(targ[in_len:], window_shape=horizon)
        fut_ts = sliding_window_view(ts_vals[in_len:], window_shape=horizon)

        nwin_full = fut_y.shape[0]
        if nwin_full == 0:
            continue

        bad_block = sliding_window_view(bad_any_row, window_shape=need)
        clean_mask = ~bad_block.any(axis=1)
        clean_mask = clean_mask[:nwin_full]

        if require_non_nan_features:
            feat_bad_row = ~np.isfinite(feat).all(axis=1)
            feat_bad_block = sliding_window_view(feat_bad_row, window_shape=in_len)
            clean_mask &= ~feat_bad_block.any(axis=1)[:nwin_full]

        feat_win = feat_win[:nwin_full][clean_mask]
        fut_y = fut_y[clean_mask]
        fut_ts = fut_ts[clean_mask]

        if step_size > 1 and len(fut_y):
            sel = slice(0, fut_y.shape[0], step_size)
            feat_win = feat_win[sel]
            fut_y = fut_y[sel]
            fut_ts = fut_ts[sel]

        nwin = feat_win.shape[0]
        if nwin == 0:
            continue

        x_list.append(feat_win.reshape(nwin, -1).astype(np.float32, copy=False))
        y_list.append(fut_y.astype(np.float32, copy=False))
        t0_list.append(fut_ts[:, 0])
        t_list.append(fut_ts)
        s_list.append(np.full(nwin, st, dtype=object))

    if not x_list:
        raise ValueError("No valid windows generated. Check data volume and feature availability.")

    x = np.vstack(x_list).astype(np.float32, copy=False)
    y = np.vstack(y_list).astype(np.float32, copy=False)
    t = np.vstack(t_list)
    t0 = pd.DatetimeIndex(np.concatenate(t0_list))
    s = np.concatenate(s_list).astype(object)

    return x, y, t0, s, t


def timestamps_to_tod_bins(t_utc: np.ndarray, tz: str = "Europe/Madrid", bins: int = 96) -> np.ndarray:
    if bins <= 0:
        raise ValueError("'bins' must be positive")
    flat = pd.to_datetime(t_utc.ravel(), utc=True).tz_convert(tz)
    mins = flat.hour.values * 60 + flat.minute.values
    idx = np.floor(mins * bins / 1440.0).astype(int)
    idx = np.clip(idx, 0, bins - 1).astype(np.int32)
    return idx.reshape(t_utc.shape)


def make_mondrian_keys_tuple(s_arr: np.ndarray, t_utc: np.ndarray, tz: str = "Europe/Madrid", tod_bins: int = 96):
    n, h = t_utc.shape
    tod = timestamps_to_tod_bins(t_utc, tz=tz, bins=tod_bins)
    keys = np.empty((n, h), dtype=object)
    for i in range(n):
        st = s_arr[i]
        for j in range(h):
            keys[i, j] = (st, int(j), int(tod[i, j]))
    return keys


def time_blocked_splits(df_all: pd.DataFrame, n_blocks: int = 2, min_train_days: int = 7):
    ts = df_all["timestamp"].sort_values().reset_index(drop=True)
    edges = np.linspace(0, len(ts), n_blocks + 1, dtype=int)
    for b in range(1, n_blocks + 1):
        start_b = ts.iloc[edges[b - 1]]
        end_idx = min(edges[b] - 1, len(ts) - 1)
        end_b = ts.iloc[end_idx]
        cutoff = start_b
        if (cutoff - ts.iloc[0]).days < min_train_days:
            continue
        train_mask = df_all["timestamp"] < cutoff
        val_mask = (df_all["timestamp"] >= cutoff) & (df_all["timestamp"] <= end_b)
        if val_mask.sum() == 0:
            continue
        yield train_mask, val_mask


class DirectMultiOutputXGB:
    def __init__(self, h: int, xgb_params: dict):
        self.h = h
        self.params = dict(xgb_params)
        self.models = []

    def fit(self, x: np.ndarray, y: np.ndarray):
        if y.shape[1] != self.h:
            raise ValueError(f"Expected y with {self.h} horizons, got {y.shape[1]}")
        self.models = []
        for j in range(self.h):
            model = xgb.XGBRegressor(**self.params)
            model.fit(x, y[:, j])
            self.models.append(model)
        return self

    def predict(self, x: np.ndarray) -> np.ndarray:
        preds = [m.predict(x) for m in self.models]
        return np.column_stack(preds).astype(np.float32)


class ResidualStore:
    def __init__(self):
        self.map = defaultdict(list)

    def push(self, key, resid: float):
        if np.isfinite(resid):
            self.map[key].append(float(resid))


class MondrianCalibrator:
    def __init__(self, alpha: float, beta: float, min_bin_n: int):
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.min_bin_n = int(min_bin_n)
        self.qL = {}
        self.qU = {}
        self.counts = {}

    def _parents(self, key):
        if not isinstance(key, tuple):
            return [key, ("__GLOBAL__",)]
        if len(key) == 3:
            st, h, _tod = key
            return [key, (st, h), (st,), ("__GLOBAL__",)]
        if len(key) == 2:
            st, h = key
            return [key, (st,), ("__GLOBAL__",)]
        if len(key) == 1:
            return [key, ("__GLOBAL__",)]
        return [key, ("__GLOBAL__",)]

    def finalize_from(self, store: ResidualStore):
        lvl3 = defaultdict(list)
        lvl2 = defaultdict(list)
        lvl1 = defaultdict(list)
        global_res = []

        for key, vals in store.map.items():
            arr = [float(v) for v in vals if np.isfinite(v)]
            if not arr:
                continue
            global_res.extend(arr)

            if isinstance(key, tuple) and len(key) == 3:
                st, h, tod = key
                lvl3[(st, h, tod)].extend(arr)
                lvl2[(st, h)].extend(arr)
                lvl1[(st,)].extend(arr)
            elif isinstance(key, tuple) and len(key) == 2:
                st, h = key
                lvl2[(st, h)].extend(arr)
                lvl1[(st,)].extend(arr)
            elif isinstance(key, tuple) and len(key) == 1:
                lvl1[key].extend(arr)

        def register(bin_key, values):
            n = len(values)
            if n < self.min_bin_n:
                return
            q = float(np.quantile(values, self.beta))
            self.qL[bin_key] = q
            self.qU[bin_key] = q
            self.counts[bin_key] = int(n)

        for k, v in lvl3.items():
            register(k, v)
        for k, v in lvl2.items():
            register(k, v)
        for k, v in lvl1.items():
            register(k, v)

        if not global_res:
            raise ValueError("No residuals collected for Mondrian calibration.")

        gq = float(np.quantile(global_res, self.beta))
        self.qL[("__GLOBAL__",)] = gq
        self.qU[("__GLOBAL__",)] = gq
        self.counts[("__GLOBAL__",)] = len(global_res)

    def interval_for(self, key, yhat: float):
        for cand in self._parents(key):
            if cand in self.qL:
                ql = self.qL[cand]
                qu = self.qU[cand]
                return yhat - ql, yhat + qu, cand
        g = self.qL[("__GLOBAL__",)]
        return yhat - g, yhat + g, ("__GLOBAL__",)


## 7) Training + Calibration Function (Full CV Loop)

This is the end-to-end training routine used in this notebook. It is intentionally explicit so reviewers can audit leakage prevention and calibration logic.

### Fold-level flow (time-blocked CV)

For each CV fold:

1. **Chronological fold split**
- Train fold contains only earlier timestamps than validation fold.

2. **Per-station scaling (train-only fit)**
- A `RobustScaler` is fitted on the fold-train slice per station.
- The fitted scaler is then applied to train+validation for that station.

3. **Feature generation with continuity**
- Train and validation are concatenated temporarily to compute lag/rolling features without boundary artifacts.
- Rows are then separated back into train and validation partitions.

4. **Station one-hot encoding (train-only fit)**
- Encoder is fitted on fold-train stations and applied to fold-validation.

5. **Window build for model fit and fold validation**
- `X_tr, Y_tr` and `X_va, Y_va` are created with the same windowing rules.
- We assert the horizon dimension is exactly 96.

6. **Model fit + residual extraction**
- A direct multi-output XGBoost model is trained on fold-train windows.
- Absolute residuals on fold-validation windows are pushed to the Mondrian residual store per key.

### Post-CV finalization

After all folds:

1. **Horizon coverage check**
- We verify Mondrian residuals exist for every horizon index `0..95`.

2. **Calibrator quantiles**
- Quantiles are computed with `beta=0.90` and `min_bin_n=100`.
- Quantiles are stored at multiple fallback levels (specific to global).

3. **Final model fit on full development data**
- Using the same locked configuration, a final model is trained on all development windows.

4. **Artifact snapshot**
- Model, transforms, calibrator maps/counts, and metadata are saved under a timestamped run directory.
- Metadata is validated to enforce `horizon = 96`.

Design intent:
- Separate model fitting from uncertainty calibration.
- Preserve strict time order.
- Keep uncertainty adaptive but robust via fallback levels.


In [ ]:
def _make_artifact_dirs(run_dir: Path) -> None:
    for sub in ["model", "transforms", "calibrator", "meta", "reports", "plots"]:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)


def _build_features_with_continuity(df_tr: pd.DataFrame, df_va: pd.DataFrame, scalers: dict[str, RobustScaler | None]):
    tr_parts = []
    va_parts = []

    for st in sorted(set(df_tr["station"].unique()) | set(df_va["station"].unique())):
        g_tr = df_tr[df_tr["station"] == st].copy()
        g_va = df_va[df_va["station"] == st].copy()
        if g_tr.empty and g_va.empty:
            continue

        g_tr["__part"] = "train"
        g_va["__part"] = "val"
        g_full = pd.concat([g_tr, g_va], ignore_index=True).sort_values("timestamp").reset_index(drop=True)

        rs = scalers.get(st)
        if rs is None:
            g_full["cons_scaled"] = g_full["consumption_clean"].astype(float).values
        else:
            g_full["cons_scaled"] = rs.transform(g_full[["consumption_clean"]].values).ravel()

        g_full = add_calendar_features(g_full, tz=CFG["tz"])
        g_full = add_target_features_in_time(g_full, value_col="cons_scaled")

        tr_parts.append(g_full[g_full["__part"] == "train"].drop(columns=["__part"]))
        va_parts.append(g_full[g_full["__part"] == "val"].drop(columns=["__part"]))

    tr_out = pd.concat(tr_parts, ignore_index=True) if tr_parts else df_tr.iloc[0:0].copy()
    va_out = pd.concat(va_parts, ignore_index=True) if va_parts else df_va.iloc[0:0].copy()
    return tr_out, va_out


def train_and_calibrate_cv_concat(
    df_train_input: pd.DataFrame,
    df_val_input: pd.DataFrame,
    cfg: dict,
    base_features: list[str],
    run_dir: Path,
):
    _make_artifact_dirs(run_dir)

    df_train_input = ensure_is_bad(df_train_input.copy())
    df_val_input = ensure_is_bad(df_val_input.copy())
    df_all = (
        pd.concat([df_train_input, df_val_input], ignore_index=True)
        .sort_values(["station", "timestamp"], kind="mergesort")
        .reset_index(drop=True)
    )

    store = ResidualStore()
    covered_h = set()

    splits = list(time_blocked_splits(df_all, n_blocks=cfg["n_blocks"], min_train_days=cfg["min_train_days"]))
    if not splits:
        raise ValueError("No valid CV folds were generated. Check n_blocks/min_train_days.")

    for fold_idx, (train_mask, val_mask) in enumerate(splits, start=1):
        fold_tr = ensure_is_bad(df_all.loc[train_mask].copy())
        fold_va = ensure_is_bad(df_all.loc[val_mask].copy())

        if fold_tr.empty or fold_va.empty:
            continue

        scalers_fold = fit_scalers_per_station(fold_tr)
        fold_tr_feat, fold_va_feat = _build_features_with_continuity(fold_tr, fold_va, scalers_fold)

        ohe_fold, ohe_cols_fold, _ = fit_station_onehot(fold_tr_feat, col="station", prefix="st__")
        fold_tr_feat = apply_station_onehot(fold_tr_feat, ohe_fold, ohe_cols_fold)
        fold_va_feat = apply_station_onehot(fold_va_feat, ohe_fold, ohe_cols_fold)

        features_fold = list(base_features) + ohe_cols_fold

        x_tr, y_tr, _t0_tr, _s_tr, _t_tr = prepare_xy_pooled_with_time(
            fold_tr_feat,
            features_fold,
            target_col="cons_scaled",
            in_len=cfg["in_length"],
            horizon=cfg["horizon"],
            step_size=cfg["step_size"],
            bad_flag_cols=("is_bad",),
        )
        x_va, y_va, _t0_va, s_va, t_va = prepare_xy_pooled_with_time(
            fold_va_feat,
            features_fold,
            target_col="cons_scaled",
            in_len=cfg["in_length"],
            horizon=cfg["horizon"],
            step_size=cfg["step_size"],
            bad_flag_cols=("is_bad",),
        )

        if y_tr.shape[1] != 96 or y_va.shape[1] != 96:
            raise ValueError(
                f"Invalid horizon dimensions in fold {fold_idx}: y_tr={y_tr.shape}, y_va={y_va.shape}."
            )

        model_fold = DirectMultiOutputXGB(h=cfg["horizon"], xgb_params=cfg["xgb_params"]).fit(x_tr, y_tr)
        yhat_va = model_fold.predict(x_va)

        keys_va = make_mondrian_keys_tuple(s_va, t_va, tz=cfg["tz"], tod_bins=96)
        for i in range(y_va.shape[0]):
            for h in range(cfg["horizon"]):
                resid = abs(float(y_va[i, h] - yhat_va[i, h]))
                store.push(keys_va[i, h], resid)
                covered_h.add(int(h))

    expected_h = set(range(cfg["horizon"]))
    if covered_h != expected_h:
        missing_h = sorted(expected_h - covered_h)
        raise ValueError(
            "Mondrian key coverage check failed. Missing horizons in calibration keys: "
            f"{missing_h}"
        )

    calibrator = MondrianCalibrator(alpha=cfg["alpha"], beta=cfg["beta"], min_bin_n=cfg["min_bin_n"])
    calibrator.finalize_from(store)

    # Final model fit on full development data
    full_scalers = fit_scalers_per_station(df_all)
    full_scaled = apply_scalers(df_all.copy(), full_scalers)

    full_feat_parts = []
    for st, g in full_scaled.groupby("station", sort=False):
        g = add_calendar_features(g, tz=cfg["tz"])
        g = add_target_features_in_time(g, value_col="cons_scaled")
        full_feat_parts.append(g)
    full_feat = pd.concat(full_feat_parts, ignore_index=True)

    ohe_final, ohe_cols_final, _ = fit_station_onehot(full_feat, col="station", prefix="st__")
    full_feat = apply_station_onehot(full_feat, ohe_final, ohe_cols_final)

    features_final = list(base_features) + ohe_cols_final

    x_all, y_all, _t0_all, _s_all, _t_all = prepare_xy_pooled_with_time(
        full_feat,
        features_final,
        target_col="cons_scaled",
        in_len=cfg["in_length"],
        horizon=cfg["horizon"],
        step_size=cfg["step_size"],
        bad_flag_cols=("is_bad",),
    )

    if y_all.shape[1] != 96:
        raise ValueError(f"Final training matrix has invalid horizon dimension: {y_all.shape}")

    final_model = DirectMultiOutputXGB(h=cfg["horizon"], xgb_params=cfg["xgb_params"]).fit(x_all, y_all)

    # Save artifacts
    joblib.dump(final_model, run_dir / "model" / "xgb_direct_multioutput.joblib")
    joblib.dump(full_scalers, run_dir / "transforms" / "station_scalers.joblib")
    joblib.dump(ohe_final, run_dir / "transforms" / "station_ohe.joblib")
    (run_dir / "transforms" / "station_ohe_columns.json").write_text(
        json.dumps(ohe_cols_final, indent=2), encoding="utf-8"
    )

    q_payload = {
        "alpha": cfg["alpha"],
        "beta": cfg["beta"],
        "min_bin_n": cfg["min_bin_n"],
        "qL": {repr(k): float(v) for k, v in calibrator.qL.items()},
        "qU": {repr(k): float(v) for k, v in calibrator.qU.items()},
    }
    counts_payload = {repr(k): int(v) for k, v in calibrator.counts.items()}

    (run_dir / "calibrator" / "quantiles.json").write_text(json.dumps(q_payload, indent=2), encoding="utf-8")
    (run_dir / "calibrator" / "counts.json").write_text(json.dumps(counts_payload, indent=2), encoding="utf-8")

    meta = {
        "run_id": run_dir.name,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "features": base_features,
        "ohe_columns": ohe_cols_final,
        "target_col": "cons_scaled",
        "in_length": cfg["in_length"],
        "horizon": cfg["horizon"],
        "step_size": cfg["step_size"],
        "alpha": cfg["alpha"],
        "beta": cfg["beta"],
        "min_bin_n": cfg["min_bin_n"],
        "tz": cfg["tz"],
        "xgb_params": cfg["xgb_params"],
        "schema": 2,
    }
    (run_dir / "meta" / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    if meta["horizon"] != 96:
        raise ValueError("Saved metadata is invalid: horizon must be 96.")

    return {
        "model": final_model,
        "calibrator": calibrator,
        "scalers": full_scalers,
        "ohe": ohe_final,
        "ohe_cols": ohe_cols_final,
        "features_final": features_final,
        "run_dir": run_dir,
        "meta": meta,
    }


## 8) Train and Calibrate


In [ ]:
run_id = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
run_dir = RUNS_ROOT / run_id

start_time = tm.time()
train_bundle = train_and_calibrate_cv_concat(
    df_train_input=df_train,
    df_val_input=df_val,
    cfg=CFG,
    base_features=BASE_FEATURES,
    run_dir=run_dir,
)
elapsed = tm.time() - start_time

print(f"Training + calibration finished in {elapsed/60:.2f} minutes")
print(f"Run directory: {train_bundle['run_dir']}")
print("Saved meta horizon:", train_bundle["meta"]["horizon"])


## 9) Holdout Validation


In [ ]:
def inverse_affine_per_station(arr_scaled: np.ndarray, s_arr: np.ndarray, scalers: dict[str, RobustScaler | None]):
    out = np.empty_like(arr_scaled, dtype=np.float32)
    for st in np.unique(s_arr):
        mask = s_arr == st
        rs = scalers.get(st)
        if rs is None:
            out[mask] = arr_scaled[mask]
            continue
        scale = float(rs.scale_[0]) if hasattr(rs, "scale_") else 1.0
        center = float(rs.center_[0]) if hasattr(rs, "center_") else 0.0
        if scale == 0.0:
            scale = 1.0
        out[mask] = arr_scaled[mask] * scale + center
    return out


def validate_holdout(train_bundle: dict, df_holdout_raw: pd.DataFrame, cfg: dict):
    model = train_bundle["model"]
    calibrator = train_bundle["calibrator"]
    scalers = train_bundle["scalers"]
    ohe = train_bundle["ohe"]
    ohe_cols = train_bundle["ohe_cols"]
    features_final = train_bundle["features_final"]

    df_holdout_raw = ensure_is_bad(df_holdout_raw.copy())

    # Scale + features by station
    hold_parts = []
    for st, g in df_holdout_raw.groupby("station", sort=False):
        rs = scalers.get(st)
        if rs is None:
            # unseen station in holdout should be skipped
            continue
        g = g.sort_values("timestamp").copy()
        g["cons_scaled"] = rs.transform(g[["consumption_clean"]].values).ravel()
        g = add_calendar_features(g, tz=cfg["tz"])
        g = add_target_features_in_time(g, value_col="cons_scaled")
        hold_parts.append(g)

    if not hold_parts:
        raise ValueError("No holdout stations matched the trained scaler set.")

    hold_feat = pd.concat(hold_parts, ignore_index=True)
    hold_feat = apply_station_onehot(hold_feat, ohe, ohe_cols)

    x_va, y_va_scaled, t0_va, s_va, t_va = prepare_xy_pooled_with_time(
        hold_feat,
        features_final,
        target_col="cons_scaled",
        in_len=cfg["in_length"],
        horizon=cfg["horizon"],
        step_size=cfg["step_size"],
        bad_flag_cols=("is_bad",),
    )

    if y_va_scaled.shape[1] != 96:
        raise ValueError(f"Holdout target horizon is invalid: {y_va_scaled.shape}")

    yhat_scaled = model.predict(x_va)

    keys_va = make_mondrian_keys_tuple(s_va, t_va, tz=cfg["tz"], tod_bins=96)
    used_h = {int(h) for h in range(keys_va.shape[1])}
    expected_h = set(range(96))
    if used_h != expected_h:
        raise ValueError("Mondrian horizon key coverage failed on holdout validation.")

    l_scaled = np.zeros_like(yhat_scaled, dtype=np.float32)
    u_scaled = np.zeros_like(yhat_scaled, dtype=np.float32)
    for i in range(yhat_scaled.shape[0]):
        for h in range(cfg["horizon"]):
            lo, hi, _src = calibrator.interval_for(keys_va[i, h], float(yhat_scaled[i, h]))
            l_scaled[i, h] = lo
            u_scaled[i, h] = hi

    y_true = inverse_affine_per_station(y_va_scaled, s_va, scalers)
    y_pred = inverse_affine_per_station(yhat_scaled, s_va, scalers)
    l_true = inverse_affine_per_station(l_scaled, s_va, scalers)
    u_true = inverse_affine_per_station(u_scaled, s_va, scalers)

    return {
        "S": s_va,
        "T0": t0_va,
        "T": t_va,
        "Y": y_true,
        "Yhat": y_pred,
        "L": l_true,
        "U": u_true,
    }


def metrics_by_horizon(results: dict):
    y = results["Y"]
    yhat = results["Yhat"]
    l = results["L"]
    u = results["U"]

    rows = []
    for h in range(y.shape[1]):
        yt = y[:, h]
        yp = yhat[:, h]
        lt = l[:, h]
        ut = u[:, h]

        mae = float(mean_absolute_error(yt, yp))
        rmse = float(np.sqrt(mean_squared_error(yt, yp)))
        picp = float(np.mean((yt >= lt) & (yt <= ut)))
        width = float(np.mean(ut - lt))

        rows.append(
            {
                "horizon": h + 1,
                "MAE": mae,
                "RMSE": rmse,
                "PICP": picp,
                "avg_width": width,
            }
        )

    metrics_df = pd.DataFrame(rows)
    summary_df = pd.DataFrame(
        {
            "metric": ["MAE_mean", "RMSE_mean", "PICP_mean", "avg_width_mean"],
            "value": [
                metrics_df["MAE"].mean(),
                metrics_df["RMSE"].mean(),
                metrics_df["PICP"].mean(),
                metrics_df["avg_width"].mean(),
            ],
        }
    )
    return metrics_df, summary_df


def per_station_metrics(results: dict):
    rows = []
    s_arr = np.asarray(results["S"])
    for st in np.unique(s_arr):
        mask = s_arr == st
        yt = results["Y"][mask].ravel()
        yp = results["Yhat"][mask].ravel()
        lt = results["L"][mask].ravel()
        ut = results["U"][mask].ravel()
        rows.append(
            {
                "station": st,
                "MAE": float(mean_absolute_error(yt, yp)),
                "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
                "PICP": float(np.mean((yt >= lt) & (yt <= ut))),
                "avg_width": float(np.mean(ut - lt)),
            }
        )
    return pd.DataFrame(rows).sort_values("station").reset_index(drop=True)


In [ ]:
results = validate_holdout(train_bundle, df_holdout, CFG)
metrics_df, summary_df = metrics_by_horizon(results)
station_df = per_station_metrics(results)

reports_dir = train_bundle["run_dir"] / "reports"
plots_dir = train_bundle["run_dir"] / "plots"
reports_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(reports_dir / "metrics_by_horizon.csv", index=False)
summary_df.to_csv(reports_dir / "overall_summary.csv", index=False)
station_df.to_csv(reports_dir / "per_station_metrics.csv", index=False)

# save a compact backtest sample for quick inspection
sample_rows = []
max_windows = min(100, results["Y"].shape[0])
for i in range(max_windows):
    st = results["S"][i]
    for h in range(CFG["horizon"]):
        sample_rows.append(
            {
                "station": st,
                "window_index": i,
                "horizon": h + 1,
                "forecast_timestamp_utc": pd.to_datetime(results["T"][i, h], utc=True),
                "y_true": float(results["Y"][i, h]),
                "y_pred": float(results["Yhat"][i, h]),
                "lower": float(results["L"][i, h]),
                "upper": float(results["U"][i, h]),
            }
        )
sample_df = pd.DataFrame(sample_rows)
sample_df.to_csv(reports_dir / "backtest_predictions_sample.csv", index=False)

# Plot 1: coverage by horizon
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(metrics_df["horizon"], metrics_df["PICP"], label="Empirical coverage")
ax.axhline(1 - CFG["alpha"], color="red", linestyle="--", label="Nominal coverage (0.90)")
ax.set_xlabel("Horizon step")
ax.set_ylabel("Coverage")
ax.set_title("Coverage by Horizon")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "coverage_by_horizon.png", dpi=150)
render_figure(fig)

# Plot 2: interval width by horizon
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(metrics_df["horizon"], metrics_df["avg_width"], color="tab:orange")
ax.set_xlabel("Horizon step")
ax.set_ylabel("Average interval width")
ax.set_title("Interval Width by Horizon")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(plots_dir / "width_by_horizon.png", dpi=150)
render_figure(fig)

# Plot 3: point error by horizon
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(metrics_df["horizon"], metrics_df["RMSE"], label="RMSE")
ax.plot(metrics_df["horizon"], metrics_df["MAE"], label="MAE")
ax.set_xlabel("Horizon step")
ax.set_ylabel("Error")
ax.set_title("Point Forecast Error by Horizon")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "error_by_horizon.png", dpi=150)
render_figure(fig)

# Plot 4: one station window example
unique_stations = sorted(pd.unique(results["S"]))
example_station = unique_stations[0]
idx_candidates = np.where(np.asarray(results["S"]) == example_station)[0]
idx = int(idx_candidates[0])

ts = pd.to_datetime(results["T"][idx], utc=True)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts, results["Y"][idx], label="True", marker="o", linewidth=1.5)
ax.plot(ts, results["Yhat"][idx], label="Pred", marker="x", linewidth=1.5)
ax.fill_between(ts, results["L"][idx], results["U"][idx], alpha=0.25, label="PI")
ax.set_title(f"Example Holdout Window - {example_station}")
ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("Consumption")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "station_example.png", dpi=150)
render_figure(fig)

print("Validation artifacts saved to:", train_bundle["run_dir"])
print("Summary metrics:")
display(summary_df)
print("First horizon rows:")
display(metrics_df.head())


## 10) Inference Demo (Latest Forecast per Station)

This section generates and saves `latest_forecast.csv` using trained artifacts only.


In [ ]:
def run_latest_inference(train_bundle: dict, df_full: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    model = train_bundle["model"]
    calibrator = train_bundle["calibrator"]
    scalers = train_bundle["scalers"]
    ohe = train_bundle["ohe"]
    ohe_cols = train_bundle["ohe_cols"]
    features_final = train_bundle["features_final"]

    df_full = ensure_is_bad(df_full.copy())

    rows = []
    for st, g in df_full.groupby("station", sort=False):
        rs = scalers.get(st)
        if rs is None:
            continue

        g = g.sort_values("timestamp").copy()
        g["cons_scaled"] = rs.transform(g[["consumption_clean"]].values).ravel()
        g = add_calendar_features(g, tz=cfg["tz"])
        g = add_target_features_in_time(g, value_col="cons_scaled")
        g = apply_station_onehot(g, ohe, ohe_cols)

        try:
            x, _y, _t0, s_arr, t_arr = prepare_xy_pooled_with_time(
                g,
                features_final,
                target_col="cons_scaled",
                in_len=cfg["in_length"],
                horizon=cfg["horizon"],
                step_size=1,
                bad_flag_cols=("is_bad",),
            )
        except ValueError:
            continue

        x_last = x[-1:]
        s_last = s_arr[-1:]
        t_last = t_arr[-1:]

        yhat_scaled = model.predict(x_last)
        keys_last = make_mondrian_keys_tuple(s_last, t_last, tz=cfg["tz"], tod_bins=96)

        l_scaled = np.zeros_like(yhat_scaled)
        u_scaled = np.zeros_like(yhat_scaled)
        for h in range(cfg["horizon"]):
            lo, hi, _src = calibrator.interval_for(keys_last[0, h], float(yhat_scaled[0, h]))
            l_scaled[0, h] = lo
            u_scaled[0, h] = hi

        yhat = inverse_affine_per_station(yhat_scaled, s_last, scalers)
        lhat = inverse_affine_per_station(l_scaled, s_last, scalers)
        uhat = inverse_affine_per_station(u_scaled, s_last, scalers)

        for h in range(cfg["horizon"]):
            rows.append(
                {
                    "station": st,
                    "horizon": h + 1,
                    "forecast_timestamp_utc": pd.to_datetime(t_last[0, h], utc=True),
                    "prediction": float(yhat[0, h]),
                    "lower": float(lhat[0, h]),
                    "upper": float(uhat[0, h]),
                }
            )

    if not rows:
        raise ValueError("No latest forecasts could be generated.")

    return pd.DataFrame(rows)


latest_forecast = run_latest_inference(train_bundle, df_pre, CFG)
latest_path = train_bundle["run_dir"] / "reports" / "latest_forecast.csv"
latest_forecast.to_csv(latest_path, index=False)

print("Saved latest forecast to:", latest_path)
latest_forecast.head(20)


## 11) Run Summary

This run produced:
- Trained model and transforms
- Mondrian quantiles with fallback levels
- Holdout metrics and diagnostic plots
- Latest forecast preview

All outputs are inside the timestamped directory printed above.
